# RAG

In [ ]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph langchain-openai langchain-core pypdf unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 27.7 MB/s eta 0:00:00


In [ ]:
# configurando chatgpt
import getpass
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [ ]:
#selecionando o embedding
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [ ]:
vector_store = InMemoryVectorStore(embeddings)

In [ ]:
#criando uma base de cohecimento
import bs4
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

file_path = "ia_intro.pdf"
loader1 = PyPDFLoader(file_path)

loader2 = WebBaseLoader(
    ["https://www.ibm.com/br-pt/think/topics/generative-ai" ]
)

docs1 = loader1.load()
docs2 = loader2.load()
docs = docs1 + docs2

In [ ]:
docs[0]

Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2011-05-17T20:46:39-04:00', 'author': 'Milani', 'moddate': '2011-05-17T20:46:39-04:00', 'source': 'ia_intro.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}, page_content='Revista Olhar Científico – Faculdades Associadas de Ariquemes – V. 01, n.2, Ago./Dez. 2010        234              \n \nInteligência Artificial: Conceitos e Aplicações \n \n \nDENNIS DOS SANTOS GOMES – dennis@w7br.com.br \nAcadêmico das Faculdades Associadas de Ariquemes- FAAr \n \nResumo \n \nO estudo da inteligência artificial teve origem há mais de dois mil anos. A busca por métodos \nou dispositivos capazes de simular o raciocínio humano vem sendo o objetivo dessa área \ndesde muito tempo. Teve início com os filósofos procurando entender como sã o realizados os \nprocessos de visão, lembranças, aprendizagem e raciocínio, várias tentativas para mecanizar a \ninteligência foram efetuadas. Desde suas orig

In [ ]:
#fazendo o splitting dos documentos
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split pdf into {len(all_splits)} sub-documents.")

Split pdf into 90 sub-documents.


In [ ]:
all_splits[0]


Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2011-05-17T20:46:39-04:00', 'author': 'Milani', 'moddate': '2011-05-17T20:46:39-04:00', 'source': 'ia_intro.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='Revista Olhar Científico – Faculdades Associadas de Ariquemes – V. 01, n.2, Ago./Dez. 2010        234              \n \nInteligência Artificial: Conceitos e Aplicações \n \n \nDENNIS DOS SANTOS GOMES – dennis@w7br.com.br \nAcadêmico das Faculdades Associadas de Ariquemes- FAAr \n \nResumo \n \nO estudo da inteligência artificial teve origem há mais de dois mil anos. A busca por métodos \nou dispositivos capazes de simular o raciocínio humano vem sendo o objetivo dessa área \ndesde muito tempo. Teve início com os filósofos procurando entender como sã o realizados os \nprocessos de visão, lembranças, aprendizagem e raciocínio, várias tentativas para mecanizar a \ninteligência foram efetuada

In [ ]:
#guardando os dados em um banco de dados
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['5b8c2723-e836-42f8-a630-d306deee6647', '89e7837d-1daa-4826-ad1c-ad04807c2510', 'c036bb8a-63d4-4d48-8134-ef33ac539211']


In [ ]:
# Create a LANGSMITH_API_KEY in Settings > API Keys
from langsmith import Client
client = Client(api_key=userdata.get('LANGSMITH_API_KEY'))
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)

In [ ]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [ ]:
# Como atualizar o prompt:

prompt.messages[0].prompt.template = '''You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say 'NAO SEI'. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:'''

In [ ]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

#definindo o workflow
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
result = graph.invoke({"question": "o que é machine learning?"})

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["answer"]}')

Context: [Document(id='36102384-d63d-45f9-9354-1ab088cabd61', metadata={'source': 'https://www.ibm.com/br-pt/think/topics/generative-ai', 'title': 'O que é a IA generativa? | IBM', 'description': 'A IA generativa é uma inteligência artificial (IA) capaz de criar conteúdo original em resposta a uma observação ou solicitação de um usuário.', 'language': 'pt', 'start_index': 9887}, page_content='Esse processo de treinamento tem um uso intenso de computação, é demorado e caro: requer milhares de unidades de processamento gráfico em clusters (GPUs) e semanas de processamento, o que custa milhões de dólares. Projetos de modelo de base de código aberto, como o Llama-2 da Meta, permitem que os desenvolvedores de IA generativa evitem essa etapa e seus custos.'), Document(id='dec3df97-f336-4146-b0a1-1ed3b19081a1', metadata={'source': 'https://www.ibm.com/br-pt/think/topics/generative-ai', 'title': 'O que é a IA generativa? | IBM', 'description': 'A IA generativa é uma inteligência artificial (IA

In [ ]:

result = graph.invoke({"question": """9. Um dos riscos clássicos em sistemas RAG é:
A) Overfitting do classificador linear durante a etapa de inferência.
B) Recuperação de contexto irrelevante ou ambíguo, degradando a resposta final.
C) Impossibilidade de uso com documentos textuais longos.
D) Necessidade obrigatória de redes convolucionais.
E) Incompatibilidade com embeddings vetoriais."""})



In [ ]:
print(f'Answer: {result["answer"]}')

Answer: B) Recuperação de contexto irrelevante ou ambíguo, degradando a resposta final.


In [ ]:

result

{'question': 'o que é machine learning?',
 'context': [Document(id='c087bd8e-ff7c-4476-b0c7-0c463e3c0396', metadata={'source': 'https://www.ibm.com/br-pt/think/topics/generative-ai', 'title': 'O que é a IA generativa? | IBM', 'description': 'A IA generativa é uma inteligência artificial (IA) capaz de criar conteúdo original em resposta a uma observação ou solicitação de um usuário.', 'language': 'pt', 'start_index': 9887}, page_content='Esse processo de treinamento tem um uso intenso de computação, é demorado e caro: requer milhares de unidades de processamento gráfico em clusters (GPUs) e semanas de processamento, o que custa milhões de dólares. Projetos de modelo de base de código aberto, como o Llama-2 da Meta, permitem que os desenvolvedores de IA generativa evitem essa etapa e seus custos.'),
  Document(id='e1362948-3afd-4cec-9c5d-547680047c29', metadata={'source': 'https://www.ibm.com/br-pt/think/topics/generative-ai', 'title': 'O que é a IA generativa? | IBM', 'description': 'A 

# Exercícios

1 - Crie um RAG capaz de resolver uma prova de IA sobre todos os assuntos já vistos durante as nossas aulas.

Utilize pelo menos 2 tipos de data loaders diferentes.

Teste o seu modelo na prova a ser aplicada.

Retorne a resposta e os documentos retornados

2 - Crie um RAG para os candidatos ao vestibular da FIAP. Use o Manual do candidato como base de conhecimento.

Teste com a seguinte pergunta:


Qual o valor da taxa de inscrição?

Retorne a resposta e os documentos retornados


